# **Домашнее задание: API-сервис сокращения ссылок.**

## **📌 Контекст задачи**:

Вам предстоит разработать сервис, который позволяет пользователям сокращать длинные ссылки, получать их аналитику и управлять ими. Основная идея — пользователь вводит длинный URL, а ваш сервис генерирует для него короткую ссылку, которую можно использовать для быстрого доступа.

**Как это работает:**

- Пользователь отправляет запрос (`POST /links/shorten`) с длинной ссылкой.

- Сервис генерирует уникальный короткий код и возвращает его пользователю.

- При открытии короткой ссылки (`GET-запрос к /{short_code}`) сервис ищет в базе данных соответствующий оригинальный URL и перенаправляет пользователя (Redirect).

**Примеры аналогичных сервисов:**
- [tinyURL](https://tinyurl.com)
- [bitly](https://bitly.com/)

---

## **🔴 Функционал сервиса:**

> Ниже представлены функции - 5 обязательных и несколько дополнительных (можно придумать свои).

### **⭕ Обязательные функции:**
1. **Создание / удаление / изменение / получение информации по короткой ссылке:**
  - `POST /links/shorten` – создает короткую ссылку.
  - `GET /links/{short_code}` – перенаправляет на оригинальный URL.
  - `DELETE /links/{short_code}` – удаляет связь.
  - `PUT /links/{short_code}` – обновляет URL (То есть, короткий адрес. Будем засчитывать и другую реализацию - когда к короткой ссылке привязывается новая длинная).
2. **Статистика по ссылке:**
  - `GET /links/{short_code}/stats`
  - Отображает оригинальный URL, возвращает дату создания, количество переходов, дату последнего использовани.
3. **Создание кастомных ссылок (уникальный alias):**
  - `POST /links/shorten` (с передачей `custom_alias`).
  - Важно проверить уникальность `alias`.
4. **Поиск ссылки по оригинальному URL:**
  - `GET /links/search?original_url={url}`
5. **Указание времени жизни ссылки:**
  - `POST /links/shorten` (создается с параметром `expires_at` в формате даты с точностью до минуты).
  - После указанного времени короткая ссылка автоматически удаляется.



### **⭕ Дополнительные функции:**

> Каждый метод должна быть отличим от других по логике и функционалу. Вы можете придумать свой вариант, но ниже будет пару наших идей.

- **Удаление неиспользуемых ссылок:**
  - К примеру, спустя по умолчанию спустя `N` дней после последнего перехода по ссылке.
  - `N` задается для всех ссылок разом.
- Отображение истории всех истекших ссылок с информацией о них.
- Группировка ссылок по проектам.
- Создание коротких ссылок для незарегистрированных пользователей.


### **⭕ Регистрация:**
Так как пользователь должен иметь возможность управлять ссылками, необходимо сделать простую регистрацию.

Изменение и удаление ссылки должно быть доступно только зарегистрированным пользователям, в то время как `GET` / `POST` - всем. Чтобы это поддерживать - храним информацию о том какой пользователь создавал ссылку (залогиненный или нет) и кладем эту информацию в базу данных.

### **⭕ База данных и кэширование:**

- Используйте БД (к примеру, `PostgreSQL`) как основное хранилище ссылок, так как позволяет легко обновлять, удалять и искать данные.

- **Обязательно используйте `Redis` или аналоги** для кэширования тех данных, где это нужно и полезно (к примеру, кэширование самых популярных ссылок). **Наличие кэширования для endpoint-ов будет оцениваться.**

- Не забудьте настроить очистку кэша при обновлении / удалении ссылки.


---

## **🔴 Формат сдачи работы:**

> Работа сдается в AnyTask.

- Ссылка на код на `GitHub`, включая `docker` или `docker-compose`.
- Cсылку на развернутый работающий сервис (можете использовать этот [хостинг](https://dashboard.render.com/)).

- `README.md` в репозитории, содержащий:
  * Описание API.
  * Примеры запросов.
  * Инструкцию по запуску.
  * Описание БД (при наличии).


---

## **🔴 Критерии оценки:**

- 5 Обязательных функций - до `7.5` баллов (`1.5` каждая)
- Наличие кэширования важных эндпоинтов - `1` балл.
- 2 Дополнительные функции - до `1.5` баллов (по `0.75` каждая).
- При реализации большей функциональности, нестандартных корректных подходах и т.п. оценивание **на усмотрение** проверяющего.

Всего за задание не может быть больше 14 баллов.


In [1]:
import string
import random
import threading
from datetime import datetime, timedelta
from typing import Optional, List

import nest_asyncio
import uvicorn
import redis
from pydantic import BaseModel
import jwt
from passlib.context import CryptContext

from fastapi import FastAPI, Depends, HTTPException, Request
from fastapi.responses import RedirectResponse
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker, Session, relationship

In [2]:
SQLALCHEMY_DATABASE_URL = "sqlite:///./shortener.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


try:
    redis_client = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
except Exception as e:
    print("Не удалось подключиться к Redis. Убедитесь, что сервер Redis запущен.")

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

In [3]:
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    hashed_password = Column(String)
    links = relationship("Link", back_populates="owner")

class Link(Base):
    __tablename__ = "links"
    id = Column(Integer, primary_key=True, index=True)
    short_code = Column(String, unique=True, index=True)
    original_url = Column(String, index=True)
    custom_alias = Column(String, unique=True, index=True, nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)
    expires_at = Column(DateTime, nullable=True)
    clicks = Column(Integer, default=0)
    last_clicked = Column(DateTime, nullable=True)
    owner_id = Column(Integer, ForeignKey("users.id"), nullable=True)
    owner = relationship("User", back_populates="links")

Base.metadata.create_all(bind=engine)

In [4]:
class UserCreate(BaseModel):
    username: str
    password: str

class Token(BaseModel):
    access_token: str
    token_type: str

class LinkCreate(BaseModel):
    original_url: str
    custom_alias: Optional[str] = None
    expires_at: Optional[datetime] = None

class LinkResponse(BaseModel):
    short_code: str
    original_url: str
    expires_at: Optional[datetime]
    
    class Config:
        orm_mode = True

class LinkUpdate(BaseModel):
    original_url: str

class LinkStats(BaseModel):
    original_url: str
    created_at: datetime
    clicks: int
    last_clicked: Optional[datetime]

C:\Users\uuu10\AppData\Local\Temp\ipykernel_6748\2676134091.py:14: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class LinkResponse(BaseModel):
d:\study\master_ai\applied_python\hw_project_3\.venv\Lib\site-packages\pydantic\_internal\_config.py:383: UserWarning: Valid config keys have changed in V2:
* 'orm_mode' has been renamed to 'from_attributes'
  warnings.warn(message, UserWarning)


In [6]:
SECRET_KEY = "secret_token_key"
ALGORITHM = "HS256"

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")


def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)


def get_password_hash(password):
    return pwd_context.hash(password)


def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(minutes=60)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)


def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username: str = payload.get("sub")
        if username is None:
            raise HTTPException(status_code=401, detail="Invalid credentials")
    except jwt.PyJWTError:
        raise HTTPException(status_code=401, detail="Invalid credentials")
        
    user = db.query(User).filter(User.username == username).first()
    if user is None:
        raise HTTPException(status_code=401, detail="User not found")
    return user


def get_optional_user(request: Request, db: Session = Depends(get_db)):
    auth_header = request.headers.get("Authorization")
    if not auth_header or not auth_header.startswith("Bearer "):
        return None
    token = auth_header.split(" ")[1]
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        if username:
            return db.query(User).filter(User.username == username).first()
    except:
        return None
    return None


In [7]:
app = FastAPI(title="URL Shortener API")

def generate_short_code(length=6):
    chars = string.ascii_letters + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

@app.post("/register", response_model=Token, tags=["Auth"])
def register(user: UserCreate, db: Session = Depends(get_db)):
    if db.query(User).filter(User.username == user.username).first():
        raise HTTPException(status_code=400, detail="Username already exists")
    hashed_password = get_password_hash(user.password)
    new_user = User(username=user.username, hashed_password=hashed_password)
    db.add(new_user)
    db.commit()
    return {"access_token": create_access_token(data={"sub": new_user.username}), "token_type": "bearer"}

@app.post("/token", response_model=Token, tags=["Auth"])
def login(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    user = db.query(User).filter(User.username == form_data.username).first()
    if not user or not verify_password(form_data.password, user.hashed_password):
        raise HTTPException(status_code=400, detail="Incorrect username or password")
    return {"access_token": create_access_token(data={"sub": user.username}), "token_type": "bearer"}